## 웹 페이지 자동 수집

### 셀레니움

- 웹 브라우저 자동으로 제어하는 파이썬 라이브러리(패키지)
- 기능
    - 브라우저 실행
    - 웹 페이지 로드(접속)
    - 입력창에 글자 입력(아이디, 패스워드 등)
    - 버튼 클릭
    - 페이지 이동
    - 화면에 표시된 텍스트 추출
    - 특정 이미지 저장 / 화면 스크린 캡처
    - 반복 자동화

- 정적인 HTML은 Beatifulsoup4, request 패키지로 수집가능
- 동적인 웹화면은 Selenium이 강력함. 정적인 웹페이지도 가능

### 웹 브라우저 실행

In [2]:
# 셀레니움 패키지 로드
from selenium import webdriver
import pandas as pd

In [3]:
driver = webdriver.Chrome()

driver.get(f'http://naver.com') # -> 열린 웹 브라우저 프로세스를 파이썬이 driver라는 변수로 관리함

![alt text](image.png)

In [5]:
# 웹 브라우저 종료
driver.quit()

#### 웹 페이지 스크래핑 실습

In [90]:
driver = webdriver.Chrome()
driver.get(f'http://127.0.0.1:5500/chapt04/products.html')

##### 요소 찾기

- 주요 요소 찾기 방식
    - `By.ID` : ID로 찾기
    - `By.CLASS_NAME` : class명으로 찾기
    - `By.TAG_NAME` : 태그명으로 찾기
    - `By.CSS_SELECTOR` : CSS 선택자 방법으로 찾기
    - `By.XPATH` : XPATH로 찾기

In [6]:
# CSS 선택자로 요소찾는 함수용 패키지 내 클래스 로드
from selenium.webdriver.common.by import By

In [7]:
# 페이지 내 하나의 데이터만 추출 (element)
title = driver.find_element(By.ID, 'title')


title.text

'상품목록'

In [8]:
title = driver.find_element(By.ID, 'search-btn')

title.text

'검색'

In [17]:
# 페이지 내 여러 요소를 동시에 추출(elements)
products = driver.find_elements(By.CLASS_NAME, 'product')

len(products)

3

In [29]:
products = driver.find_elements(By.CLASS_NAME, 'product')
for product in products:
    print(product.text)

키보드 30000
마우스 15000
모니터 250000


##### CSS 선택자료 요소 추출


In [34]:
products = driver.find_elements(By.CSS_SELECTOR,'.product')

for product in products:
    print(product)
    name = product.find_element(By.CSS_SELECTOR, '.name').text
    price = product.find_element(By.CSS_SELECTOR, '.price').text
    print(name, price)

<selenium.webdriver.remote.webelement.WebElement (session="c005ad738b222a5cb36871011fccf4a1", element="f.1F97DE80A84A36187DAA2649EC02C134.d.91977C36B7635F3D344347107139F2B0.e.1057")>
키보드 30000
<selenium.webdriver.remote.webelement.WebElement (session="c005ad738b222a5cb36871011fccf4a1", element="f.1F97DE80A84A36187DAA2649EC02C134.d.91977C36B7635F3D344347107139F2B0.e.1060")>
마우스 15000
<selenium.webdriver.remote.webelement.WebElement (session="c005ad738b222a5cb36871011fccf4a1", element="f.1F97DE80A84A36187DAA2649EC02C134.d.91977C36B7635F3D344347107139F2B0.e.1063")>
모니터 250000


##### 입력창 입력

In [67]:
keyword_input = driver.find_element(By.ID, 'keyword')
keyword_input.send_keys('USB');

In [68]:
keyword_input.clear()

In [69]:
keyword_input.send_keys('테슬라')

In [70]:
keyword_str = driver.find_element(By.ID, 'keyword')

keyword_str.get_attribute('value')

'테슬라'

##### 버튼 클릭

In [72]:
button = driver.find_element(By.ID, 'search-btn')
button.click()

##### 명시적 대기

- 링크 클릭으로 다른 페이지 이동, 검색 후 결과 대기까지 시간이 소요
- `time.sleep(2)` 2초동안 대기 등의 파이썬 기본 함수가 제약사항이 많음
- 특정 조건이 만족할 때까지 대기하는 로직을 사용해야 함

In [88]:
# 명시적 대기를 위한 패키지 로드
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [103]:
# 여러 작업 수행 후

wait = WebDriverWait(driver, 10) # 10초까지 대기

title = wait.until(
    EC.presence_of_element_located((By.ID, 'title'))
)

title.text

'상품목록'

##### 데이터 수집하기

In [110]:
products = driver.find_elements(By.CLASS_NAME, 'product')

result = []

for product in products:
    name = product.find_element(By.CSS_SELECTOR,'.name').text
    price = product.find_element(By.CLASS_NAME,'price').text

    result.append({'name' : name, 'price' : price})

In [111]:
result

[{'name': '키보드', 'price': '30000'},
 {'name': '마우스', 'price': '15000'},
 {'name': '모니터', 'price': '250000'}]

In [112]:
df = pd.DataFrame(result)

df

,name,price
0,키보드,30000
1,마우스,15000
2,모니터,250000


In [113]:
df.to_csv('../data/scrap_products.csv', encoding='utf=8')

##### 정리

- 웹 스크래핑 시 필요 요소
    - 웹 드라이버로 웹사이트 오픈
    - 필요 요소 찾기
    - 입력태그 찾기
    - 버튼 요소 찾기
    - 입력하고 버튼 클릭해서 동작
    - 데이터 추출
    - 화면이 갱신되기까지 명시적 대기

#### 실습

##### 멜론 차트 TOP100 스크래핑

In [153]:
# 페이지 전환없는 스크래핑용 패키지 로드
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [154]:
driver = webdriver.Chrome()
wait = WebDriverWait(driver, 10)

driver.get('https://www.melon.com/chart/index.htm')

company_name = wait.until(
    EC.presence_of_element_located((By.ID, 'footer'))
)

##### 차트의 곡 행 가져오기

In [ ]:
# css 선택자 입력 시 태그 + .css 동일한 기능
#songs = driver.find_elements(By.CSS_SELECTOR,'.lst50, .lst100')
songs = driver.find_elements(By.CSS_SELECTOR,'tr.lst50, tr.lst100')

datas = []


#len(songs)
for song in songs:
    #노래 순위
    rank = song.find_element(By.CSS_SELECTOR,'.rank').text
    #노래 제목
    title = song.find_element(By.CSS_SELECTOR,'.rank01 > span > a').text
    #가수명
    singer = song.find_element(By.CSS_SELECTOR, '.rank02 a').text
    #앨범명
    album = song.find_element(By.CSS_SELECTOR, '.rank03 a').text
    #좋아요 개수
    like = song.find_element(By.CSS_SELECTOR,'.cnt ').text
    # MV 링크 , 복잡함.
    # MV = song.find_element(By.CSS_SELECTOR,'.')

    #단계상승하락
    rank_ud = song.find_element(By.CSS_SELECTOR, '.rank_wrap').get_attribute('title')

    #이미지 썸네일 주소
    #img_url = song.find_element(By.CSS_SELECTOR, '.image_typeAll  img').get_attribute('src')

    datas.append({
        '순위' : rank,
        '순위변동' : rank_ud,
        '노래제목' : title,
        '가수명' : singer,
        '앨범명' : album,
        '좋아요개수' : like
    })


    #print(rank, title, singer, album, like)

In [ ]:
# css 선택자 입력 시 태그 + .css 동일한 기능
#songs = driver.find_elements(By.CSS_SELECTOR,'.lst50, .lst100')
songs = driver.find_elements(By.CSS_SELECTOR,'tr.lst50, tr.lst100')


datas = []

#len(songs)
for song in songs:
    #노래 순위
    rank = song.find_element(By.CSS_SELECTOR,'.rank').text
    #노래 제목
    title = song.find_element(By.CSS_SELECTOR,'.rank01 > span > a').text
    #가수명
    singer = song.find_element(By.CSS_SELECTOR, '.rank02 a').text
    #앨범명
    album = song.find_element(By.CSS_SELECTOR, '.rank03 a').text
    #좋아요 개수
    like = song.find_element(By.CSS_SELECTOR,'.cnt ').text

    datas.append({
        '순위' : rank,
        '노래제목' : title,
        '가수명' : singer,
        '앨범명' : album,
        '좋아요개수' : like
    })

In [141]:
driver.quit()

In [145]:
df_melon100 = pd.DataFrame(datas)
df_melon100

,순위,노래제목,가수명,앨범명,좋아요개수
0,1,퇴사할게여 (Narr. 기안84),소연 (SOYEON),끝내주는 인생,"39,452"
1,2,LOVE ATTACK,RESCENE (리센느),SCENEDROME,"153,113"
2,3,갑자기,아이오아이 (I.O.I),I.O.I 3rd MINI ALBUM [I.O.I : LOOP],"80,168"
3,4,Deja Vu,RESCENE (리센느),Dearest,"58,916"
4,5,Pretty Girl,RESCENE (리센느),Pretty Girl - Special Single,"66,134"
...,...,...,...,...,...
95,96,STYLE,Hearts2Hearts (하츠투하츠),STYLE,"66,839"
96,97,Vitamin ME,프로미스나인,Glow ME,"20,246"
97,98,Less than a Lover,제니 (JENNIE),Less than a Lover,"27,095"
98,99,REBEL HEART,IVE (아이브),IVE EMPATHY,"122,856"


In [151]:

df_melon100.to_csv('../data/26년10월1일노래순위.csv', encoding='utf=8')